In [2]:
# !pip install -q ipdb
# import ipdb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, ToTensor, Normalize
import matplotlib.pyplot as plt
from random import randint

In [ ]:
tr = Compose([
    ToTensor(),
    Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
train_dataset = CIFAR10('/.', train=True, download=True, transform=tr)
test_dataset = CIFAR10('/.', train=False, download=False, transform=tr)

clases = ('avión', 'auto', 'ave', 'gato', 'venado',
           'perro', 'rana', 'caballo', 'barco', 'camión')

In [ ]:
class InceptionModule(nn.Module):
  def __init__(self, 
               in_channels,
               n_classes, 
               ch_3x3_reduce=96, 
               ch_5x5_reduce=16,
               ch_3x3=128,
               ch_5x5=32,
               ch_pool_proj=32,
               ch_1x1=64
    ):
    super(InceptionModule, self).__init__()

    #Branch 1 
    self.conv_1p1_c1 = nn.Conv2d(in_channels, ch_3x3_reduce, (1,1), stride=1, padding=0)
    self.conv_3p3 = nn.Conv2d(ch_3x3_reduce, ch_3x3, (3,3), stride=1, padding=1)

    #Branch 2
    self.conv_1p1_c2 = nn.Conv2d(in_channels, ch_5x5_reduce, (1,1), stride=1, padding=0)
    self.conv_5p5 = nn.Conv2d(ch_5x5_reduce, ch_5x5, (5,5), stride=1, padding=2)

    #Branch 3
    self.pool = nn.MaxPool2d((3,3), stride=1, padding=1)
    self.conv_1p1_d3 = nn.Conv2d(in_channels, ch_pool_proj, (1,1), stride=1, padding=0)

    #Branch 4
    self.conv_1p1_d4 = nn.Conv2d(in_channels, ch_1x1, (1,1), stride=1, padding=0)

    #logit/hidden 
    self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
    self.fc = nn.Linear(256,n_classes)


  def forward(self, x):
    # Calcula la salida como un tensor con cantidad de canales de
    # salida dado por ch_3x3 + ch_5x5 + ch_pool_proj + ch_1x1
    #Branch 1
    x1_0 = self.conv_1p1_c1(x)
    x1 = self.conv_3p3(x1_0)

    #Branch 2
    x2_0 = self.conv_1p1_c2(x)
    x2 = self.conv_5p5(x2_0)

    #Branch 3
    x_pool = self.pool(x)
    x3 = self.conv_1p1_d3(x_pool)

    #Branch 4
    x4 = self.conv_1p1_d4(x)

    x = torch.cat([x1,x2,x3,x4], dim=1)  #(B, 256, H, W)

    #OUT
    pooled = self.global_avg_pool(x) #(B,256,1,1)
    hidden = torch.flatten(pooled, 1) #(B,256)

    logits = self.fc(hidden)  
    
    return {
        "logits": logits,
        "hidden": hidden}

In [1]:
x = torch.randn(4, 3, 32, 32)

model = InceptionModule(
    in_channels=3,
    n_classes=10
)

out = model(x)

print(out["hidden"].shape)
print(out["logits"].shape)

NameError: name 'torch' is not defined